In [2]:
!pip install -q google-generativeai

In [3]:
import google.generativeai as genai
from google.colab import userdata
import numpy as np

genai.configure(api_key=userdata.get("GEMINI_KEYS"))

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
result = genai.embed_content(
    model="models/gemini-embedding-001",
    content="React developer with payment gateway experience.",
    output_dimensionality=768
)

embeddings = result['embedding']
print("Type:", type(embeddings))
print("Length:", len(embeddings))
print("First few elements:", embeddings[:8])

Type: <class 'list'>
Length: 768
First few elements: [-0.0033286544, -0.013729486, 0.027328545, -0.07900387, -0.02442697, 0.017155312, 0.0491243, 0.0122431405]


In [5]:
sentences = [
    "React developer with payment gateway experience.",
    "Built UPI checkout flow for Razorpay using Next.js and Node",
    "Cardiologist with 12 years experience at AIIMS Delhi",
]

embeddings = []

for s in sentences:
  emb = genai.embed_content(
      model="models/gemini-embedding-001",
      content=s,
      output_dimensionality=768
  )['embedding']
  embeddings.append(emb)


embeddings = np.array(embeddings)
print(embeddings.shape)


(3, 768)


In [12]:
!pip install -q google-genai faiss-cpu numpy

import os
import json
import time
import numpy as np
import faiss
from google import genai
from google.genai import types

# Securely grab the API key from Colab secrets or environment variables
from google.colab import userdata
try:
    API_KEY = userdata.get('GEMINI_KEYS')
except:
    API_KEY = os.environ.get("GEMINI_KEYS", "YOUR_API_KEY_HERE")

client = genai.Client(api_key=API_KEY)

In [7]:
# Create the resumes directory
os.makedirs("resumes", exist_ok=True)

mock_resumes = {
    "resume_01.txt": "John Doe. Senior Software Engineer with 8 years of experience in Python, Django, AWS, and distributed systems. Expert in backend optimization.",
    "resume_02.txt": "Jane Smith. UI/UX Designer specializing in Figma, Adobe XD, user research, and wireframing for mobile apps. 4 years experience.",
    "resume_03.txt": "Alice Johnson. Digital Marketing Specialist. Expert in SEO, Google Ads, Meta Blueprints, and growth hacking. Increased conversion by 40%.",
    "resume_04.txt": "Bob Lee. Recent Computer Science Graduate from State Tech. Proficient in Java, C++, and basic SQL. Looking for entry-level Software Engineer roles. Projects include a basic chat app.",
    "resume_05.txt": "Charlie Brown. HR Generalist with 5 years in talent acquisition, employee relations, and onboarding frameworks. Proficient in Workday.",
    "resume_06.txt": "Diana Prince. Data Scientist. 3 years experience building machine learning models using Scikit-Learn, Pandas, and TensorFlow. Strong background in statistical analysis.",
    "resume_07.txt": "Evan Wright. Financial Analyst. Certified CFA Level II. Experienced in corporate budgeting, forecasting, and Excel macros. 6 years in investment banking.",
    "resume_08.txt": "Fiona Gallagher. Customer Success Manager. 4 years handling enterprise clients, reducing churn by 15%, and managing Zendesk tickets.",
    "resume_09.txt": "George Costanza. Real Estate Agent. Experienced in property sales, commercial leasing, and client contract negotiations. 10 years in the NYC market.",
    "resume_10.txt": "Hannah Abbott. DevOps Engineer. Expert in Docker, Kubernetes, CI/CD pipelines (Jenkins, GitHub Actions), and Terraform Infrastructure as Code.",
    "resume_11.txt": "Ian Malcolm. Chaos Mathematician and Data Analyst. Specialized in predictive modeling, R programming, and identifying anomalies in massive datasets.",
    "resume_12.txt": "Julia Roberts. Executive Assistant. 7 years managing calendar logistics, travel arrangements, C-suite communications, and event planning.",
    "resume_13.txt": "Kevin Mitnick. Cybersecurity Specialist. Experience in penetration testing, network security protocols, firewalls, and incident response management.",
    "resume_14.txt": "Laura Croft. Field Archaeologist and Museum Curator. Managed historical digs, artifact archiving, and historical research documentation.",
    "resume_15.txt": "Michael Scott. Regional Sales Manager. 15 years leading B2B paper sales teams. Expert in relationship building, client retention, and motivational speaking.",
    "resume_16.txt": "Nancy Drew. Investigative Journalist & Content Writer. Skilled in deep research, investigative reporting, SEO copy editing, and long-form essay writing.",
    "resume_17.txt": "Oscar Martinez. Senior Accountant. CPA certified. 12 years managing general ledgers, corporate tax preparation, and internal audits.",
    "resume_18.txt": "Peter Parker. Freelance Photojournalist and Action Photographer. Digital editing, high-speed sports photography, and Adobe Lightroom master.",
    "resume_19.txt": "Quinn Fabray. Nurse Practitioner. Licensed MSN with 5 years in emergency room patient care, triage management, and clinical workflows.",
    "resume_20.txt": "Ray Stantz. Mechanical Engineer. Specialized in CAD modeling, thermal dynamics, and custom hardware prototyping. 8 years prototyping experimental equipment.",
    "resume_21.txt": "Sam Wilson. Veteran Career Transitioner. Former military logistics officer seeking an entry-level operations role. Strong leadership and supply chain coordination skills."
}

for filename, content in mock_resumes.items():
    with open(f"resumes/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print(f"Successfully generated {len(mock_resumes)} resumes in './resumes/' folder.")

Successfully generated 21 resumes in './resumes/' folder.


In [17]:
def get_embedding(text: str) -> list:
    """Helper function to fetch embeddings using models/gemini-embedding-001."""
    response = client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=text
    )
    return response.embeddings[0].values

def build_embeddings():
    """Reads all resumes, creates metadata summaries, generates embeddings, and saves to disk."""
    resume_folder = "resumes"
    vectors = []
    metadata = []

    files = [f for f in os.listdir(resume_folder) if f.endswith('.txt')]

    print("Processing and embedding resumes...")
    for idx, filename in enumerate(files):
        with open(os.path.join(resume_folder, filename), "r", encoding="utf-8") as f:
            content = f.read().strip()

        # For small resumes, the full string is our summary string.
        # For massive files, you would pass this to Gemini to generate a 2-line summary first.
        summary_str = content

        # Get vector from Gemini
        vector = get_embedding(summary_str)
        vectors.append(vector)

        # Save structural metadata
        metadata.append({
            "id": idx,
            "filename": filename,
            "summary": summary_str
        })

    # Convert vectors to NumPy array
    vectors_np = np.array(vectors, dtype=np.float32)

    # Save files to disk
    np.save("resumes.npy", vectors_np)
    with open("resumes_meta.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4)

    print(f"Saved {len(vectors)} vectors to 'resumes.npy' and metadata to 'resumes_meta.json'.")

# Execute the pipeline execution
build_embeddings()

Processing and embedding resumes...
Saved 21 vectors to 'resumes.npy' and metadata to 'resumes_meta.json'.


In [18]:
def search_candidates_numpy(jd: str, top_k: int = 5, threshold: float = 0.55):
    """Searches resumes using raw NumPy array operations via Cosine Similarity."""
    if not os.path.exists("resumes.npy") or not os.path.exists("resumes_meta.json"):
        return "Error: Missing embeddings or metadata. Run build_embeddings() first."

    vectors = np.load("resumes.npy")
    with open("resumes_meta.json", "r") as f:
        metadata = json.load(f)

    # Embed Job Description
    jd_vector = np.array(get_embedding(jd), dtype=np.float32)

    # Cosine Similarity = (A • B) / (||A|| * ||B||)
    # text-embedding-004 output is normalized to unit length L2, so standard dot product equals cosine similarity
    dot_products = np.dot(vectors, jd_vector)
    norms_vectors = np.linalg.norm(vectors, axis=1)
    norm_jd = np.linalg.norm(jd_vector)
    similarities = dot_products / (norms_vectors * norm_jd)

    # Filter and Rank
    ranked_indices = np.argsort(similarities)[::-1]

    results = []
    for idx in ranked_indices:
        score = float(similarities[idx])
        if score < threshold:
            continue
        meta = metadata[idx].copy()
        meta["score"] = score
        results.append(meta)

    return results[:top_k] if results else "No strong matches found."


def search_candidates_faiss(jd: str, top_k: int = 5, threshold: float = 0.55):
    """Searches resumes using an in-memory Flat L2/Inner Product FAISS Index."""
    if not os.path.exists("resumes.npy") or not os.path.exists("resumes_meta.json"):
        return "Error: Missing embeddings. Run build_embeddings() first."

    vectors = np.load("resumes.npy")
    with open("resumes_meta.json", "r") as f:
        metadata = json.load(f)

    jd_vector = np.array([get_embedding(jd)], dtype=np.float32)

    # Normalize vectors to use IndexFlatIP for exact Cosine Similarity
    faiss.normalize_L2(vectors)
    faiss.normalize_L2(jd_vector)

    dimension = vectors.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(vectors)

    # Search index
    scores, indices = index.search(jd_vector, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1 or score < threshold:
            continue
        meta = metadata[idx].copy()
        meta["score"] = float(score)
        results.append(meta)

    return results if results else "No strong matches found."


def generate_fit_reason(jd: str, resume_summary: str) -> str:
    """STAGE 4 Bonus: Asks Gemini to provide a crisp, one-liner explanation for the UI."""
    prompt = f"""
    You are an expert HR Talent Scout. Given the following Job Description and a Candidate's resume summary, provide exactly a ONE-LINE reason explaining why this candidate is a semantic fit. Be specific to their skills.

    Job Description: {jd}
    Candidate Resume: {resume_summary}

    One-line Reason:"""

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
        )
        return response.text.strip()
    except Exception as e:
        return "Match verified via embedding proximity vector analysis."

In [20]:
# Define Test JDs
jd_tech = "Looking for a Cloud Infrastructure expert who can deploy Kubernetes clusters and configure infrastructure as code using Terraform."
jd_nontech = "Seeking a media relations professional or investigative reporter skilled in deep research, long-form writing, and editing content."
jd_nonsense = "Looking for an astronaut qualified in underwater basket weaving and zero-gravity pancake flipping."

test_cases = [
    ("TECH JOB DESCRIPTION", jd_tech),
    ("NON-TECH JOB DESCRIPTION", jd_nontech),
    ("NONSENSE JOB DESCRIPTION", jd_nonsense)
]

for label, jd in test_cases:
    print("="*80)
    print(f" TESTING: {label}")
    print(f"JD text: '{jd}'\n")

    # --- Numpy Search ---
    start_np = time.perf_counter()
    res_np = search_candidates_numpy(jd, top_k=3)
    end_np = time.perf_counter()
    time_np = end_np - start_np

    # --- FAISS Search ---
    start_faiss = time.perf_counter()
    res_faiss = search_candidates_faiss(jd, top_k=3)
    end_faiss = time.perf_counter()
    time_faiss = end_faiss - start_faiss

    print(f" Numpy Time: {time_np:.6f} seconds")
    print(f" FAISS Time: {time_np:.6f} seconds\n")

    # Validate Match Consistency
    if isinstance(res_np, str) or isinstance(res_faiss, str):
        if res_np == res_faiss:
            print(f" Both engines agreed: {res_np}\n")
        else:
            print(f" Discrepancy found! Numpy: {res_np} | FAISS: {res_faiss}\n")
    else:
        np_filenames = [r['filename'] for r in res_np]
        faiss_filenames = [r['filename'] for r in res_faiss]
        if np_filenames == faiss_filenames:
            print(" Cross-Engine Check: SUCCESS (Both engines returned matching top-K candidates)\n")
        else:
            print(" Cross-Engine Check: MISMATCH in rankings.\n")

        print("🔝 TOP CANDIDATE DETAILS & AI EXPLANATION:")
        for idx, match in enumerate(res_np):
            print(f"  {idx+1}. {match['filename']} (Score: {match['score']:.4f})")
            # Run Stage 4 Explainer on the top match
            if idx == 0:
                reason = generate_fit_reason(jd, match['summary'])
                print(f"      Gemini Fit Reason: {reason}")
        print("\n")

 TESTING: TECH JOB DESCRIPTION
JD text: 'Looking for a Cloud Infrastructure expert who can deploy Kubernetes clusters and configure infrastructure as code using Terraform.'

 Numpy Time: 0.341499 seconds
 FAISS Time: 0.341499 seconds

 Cross-Engine Check: SUCCESS (Both engines returned matching top-K candidates)

🔝 TOP CANDIDATE DETAILS & AI EXPLANATION:
  1. resume_10.txt (Score: 0.7000)
      Gemini Fit Reason: Hannah is an expert in Kubernetes and Terraform Infrastructure as Code, directly matching the requirements for deploying Kubernetes clusters and configuring infrastructure as code.
  2. resume_01.txt (Score: 0.6073)
  3. resume_04.txt (Score: 0.5640)


 TESTING: NON-TECH JOB DESCRIPTION
JD text: 'Seeking a media relations professional or investigative reporter skilled in deep research, long-form writing, and editing content.'

 Numpy Time: 0.190987 seconds
 FAISS Time: 0.190987 seconds

 Cross-Engine Check: SUCCESS (Both engines returned matching top-K candidates)

🔝 TOP CANDI